In [ ]:
import pandas as pd
import numpy as np
import keras
from sklearn.metrics import r2_score,root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,StandardScaler
import matplotlib.pyplot as plt
from climQMBC.methods import DQM
import itertools
from sklearn import svm
from sanfis import SANFIS
from sanfis.datagenerators import sanfis_generator
import torch
from sklearn.model_selection import GridSearchCV
from keras_tuner import RandomSearch,GridSearch
from keras_tuner.engine.hyperparameters import HyperParameters
import os
import shutil
import pywt
import itertools
import seaborn as sns

import tensorflow.keras.backend as K
import tensorflow as tf
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.metrics import Metric
from tqdm import tqdm
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf
import pickle

In [ ]:
class NashSutcliffeEfficiency(Metric):
    def __init__(self, name='nse', **kwargs):
        super(NashSutcliffeEfficiency, self).__init__(name=name, **kwargs)
        self.numerator = self.add_weight(name='numerator', initializer='zeros')
        self.denominator = self.add_weight(name='denominator', initializer='zeros')
        self.count = self.add_weight(name='count', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        numerator = tf.reduce_sum(tf.square(y_true - y_pred))
        denominator = tf.reduce_sum(tf.square(y_true - tf.reduce_mean(y_true)))
        self.numerator.assign_add(numerator)
        self.denominator.assign_add(denominator)
        self.count.assign_add(1)

    def result(self):
        nse = 1 - (self.numerator / (self.denominator+1e-9))
        return nse

    def reset_states(self):
        self.numerator.assign(0)
        self.denominator.assign(0)
        self.count.assign(0)
def closest_grid(x,y):
    x = x.astype(np.float32)
    y = y.astype(np.float32)
    sums={}
    for col in x.columns:
        
        cc=np.corrcoef(x[col],y)[0,1]
        nse=r2_score(x[col],y)
        print(f'Grid {col} Correlation with Target :{round(cc,3)} NSE score: {round(nse,3)}, Overall results:')
        if nse>0.2 and cc>0.2:
            print("High CC, High NSE: Excellent data/model.")
        elif cc>0.2 and nse<0.2:
            print("High CC, Low NSE: Good trend capture but poor magnitude prediction.")
        elif cc<0.2 and nse<0.2:
            print("Low CC, Low NSE: Poor performance overall")
        sums[col]=round(nse+cc,3)
    max_key = max(sums, key=sums.get)
    
    print(f'Grid with maximum value: {max_key}, Value: {sums[max_key]} Added to Inputs')
    return max_key,x[max_key].values
    
def clear_tuner_directory(directory):
    if os.path.exists(directory):
        shutil.rmtree(directory)
        
def prepare_lstm_input(x, y, timesteps):
    x_lstm = []
    y_lstm = []
    for i in range(len(x) - timesteps):
        x_lstm.append(x[i:i+timesteps,:])
        y_lstm.append(y[i+timesteps])
    return np.array(x_lstm), np.array(y_lstm)

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


In [ ]:
station='sf' #Specify the station name by choosing the appropriate folder sheet name. In this case, it is 'sf' standing for San Francisco. you can choose among [sf,mi,ph,ch]
gcms=pd.read_excel(f".\Sample Date\historical Inputs\{station}\PS_inputs.xlsx") 
gcms['date'] = pd.date_range(start='1981-01-01', end='2014-12-01', freq='MS')
gcms['season'] = gcms['date'].dt.month.map(get_season)
gcms['monthsin']=np.sin(2 * np.pi * gcms['date'].dt.month / 12)
gcms['monthcos']=np.cos(2 * np.pi * gcms['date'].dt.month / 12)
seasons=pd.get_dummies(gcms, columns=['season']).set_index('date')

gcms.set_index('date',inplace=True)
observations= gcms['obs']
prism=gcms['prism']
time=gcms.index
selected_variants= ['r1i1p1f1','r6i1p1f1']
selected_columns = {v: [c for c in gcms.columns if v in c] for v in selected_variants}


In [ ]:
#without details
thresholds = [5 ]
wavelet = 'db4'
level = 4
selected_level=3
fig, axes = plt.subplots(nrows=level, ncols=len(thresholds), figsize=(40, 15), sharex=True, sharey=True)
fig.suptitle('Wavelet Denoising Approximations at Different Thresholds and Levels',  fontsize=38, fontdict={'color': 'Black', 'font': 'Times New Roman'})
if len(thresholds) == 1:
    axes = np.expand_dims(axes, axis=1)
for col, threshold in enumerate(thresholds):
    coeffs = pywt.wavedec(prism, wavelet, level=level, mode='periodization')
    thresholded_coeffs = [coeffs[0]] + [pywt.threshold(c, threshold, mode='soft') for c in coeffs[1:]]
    
    approximations = []
    for i in range(level):
        coeffs_copy = thresholded_coeffs.copy()
        coeffs_copy[i+1:] = [np.zeros_like(c) for c in coeffs[i+1:]]
        approximations.append(pywt.waverec(coeffs_copy, wavelet, mode='periodization')[:len(prism.values)])
    
    for row, approx in enumerate(approximations):
        
        axes[row,col].plot(time, approx, label=f'Level {row+1} Threshold: {threshold}', color='green')
        axes[row, col].set_title(f'Approximation \n Level {row+1} Threshold: {threshold}', fontdict={'family': 'Times New Roman', 'size': 20})
        axes[row, col].tick_params(axis='both', which='major', labelsize=15, labelcolor='black', labelrotation=45, labelbottom=True, labeltop=False, labelleft=True, labelright=False, labelfontfamily='Times New Roman')
        
        axes[row,col].legend()
        if row+1 == 3:
            gcms[f'approx_{row+1}_threshold_{threshold}'] = approx
        if row == 3:
            axes[row,col].set_xlabel('Time')
        if col == 0:
            axes[row,col].set_ylabel('Value')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.style.use('default')
plt.show()

selected_targets = {v: [c for c in gcms.columns if v in c] for v in ['approx']}

In [ ]:
tf.config.experimental.enable_op_determinism()
tf.random.set_seed(1)
validation_dc_list_ffnn={}
calibration_dc_list_ffnn={}
validation_cc_list_ffnn={}
calibration_cc_list_ffnn={}
validation_rmse_list_ffnn={}
calibration_rmse_list_ffnn={}
batch_dict={}
learning_rate = [0.001]
learning_rate_dict={}
batch_size = [64]
for batch in batch_size:
    for lr in learning_rate:
        x = gcms[sum(selected_columns.values(), [])]
        x['monthsin'],x['monthcos']=gcms['monthsin'],gcms['monthcos']
        y = gcms[selected_targets['approx']]

        
        xtrain, xtest, ytrain, ytest = train_test_split(x, y, train_size=0.7, shuffle=False)
        index_train, index_test = train_test_split(gcms.index, train_size=0.7, shuffle=False)
        ytrain = np.array(ytrain).reshape(-1, 1)
        ytest = np.array(ytest).reshape(-1, 1)
        
        x_scaler = StandardScaler()
        y_scaler = StandardScaler()
        xtrain = x_scaler.fit_transform(xtrain)
        xtest = x_scaler.transform(xtest)
        
        ytrain = y_scaler.fit_transform(ytrain).reshape(-1)
        ytest = y_scaler.transform(ytest).reshape(-1)
        print("inputs-validation set normalized std dev:",np.std(ytest))
        print(f'Input Shape: {xtrain.shape}\n target shape: {ytrain.shape}')
        
        ffnn_model = keras.models.Sequential([
                keras.layers.Dense(16, activation=keras.activations.selu, kernel_regularizer=keras.regularizers.L1L2(0.005),kernel_initializer=keras.initializers.he_normal(seed=1)),
                keras.layers.Dense(32, activation=keras.activations.selu),
                keras.layers.Dense(16, activation=keras.activations.selu),
                keras.layers.Dropout(0.2,seed=1),
                keras.layers.Dense(1)

            ])
        ffnn_model.compile(loss=keras.losses.mean_absolute_error, optimizer=keras.optimizers.Adam(learning_rate=lr), metrics=[NashSutcliffeEfficiency()])
        early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)
        lr_scheduler = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=30, verbose=1, min_lr=1e-5)
        
        history = ffnn_model.fit(x=xtrain, y=ytrain, epochs=80, batch_size=batch, validation_split=0.3,callbacks=[lr_scheduler,early_stopping])
        ypred_ffnn = ffnn_model.predict(xtest)
        ypred_train_ffnn = ffnn_model.predict(xtrain)

        plt.figure(figsize=(5, 5))
        plt.plot(history.history['loss'], label=f'loss with Batch: {batch}')
        plt.plot(history.history['val_loss'], label='val_Loss')
        plt.legend()
        

        dc_c = r2_score(y_true=ytrain, y_pred=ypred_train_ffnn)
        coer_c = np.corrcoef(ypred_train_ffnn.reshape(ypred_train_ffnn.shape[0]), ytrain.reshape(ytrain.shape[0]))[0, 1]
        rmse_c = root_mean_squared_error(y_true=y_scaler.inverse_transform(ytrain.reshape(-1,1)), y_pred=y_scaler.inverse_transform(ypred_train_ffnn.reshape(-1,1)))
    
        
        
        plt.figure(figsize=(15, 5))
        plt.plot( ypred_ffnn, label=f'Prediction', color='green')
        plt.plot( ytest, label=f'PRSIM', color='blue',alpha=0.5)
        plt.legend()
        dc_v = r2_score(y_true=ytest, y_pred=ypred_ffnn)
        batch_dict[batch]=dc_v
        learning_rate_dict[lr]=dc_v
        coer_v = np.corrcoef(ypred_ffnn.reshape(ypred_ffnn.shape[0]), ytest.reshape(ytest.shape[0]))[0, 1]
        rmse_v = root_mean_squared_error(y_true=y_scaler.inverse_transform(ytest.reshape(-1,1)), y_pred=y_scaler.inverse_transform(ypred_ffnn.reshape(-1,1)))
        print(f'Batch size: {batch} DC Validation:{round(dc_v, 3)}, CC Validation:{round(coer_v, 3)}')
        print(f'DC Calibration:{round(dc_c, 3)}, CC Calibration:{round(coer_c, 3)}')
        validation_dc_list_ffnn['dc_v'] = round(dc_v, 3)
        calibration_dc_list_ffnn['dc_c'] = round(dc_c, 3)
        validation_cc_list_ffnn['cc_v']=round(coer_v, 3)
        calibration_cc_list_ffnn['cc_c']=round(coer_c, 3)
        validation_rmse_list_ffnn['rmse_d']=round(rmse_v, 3)
        calibration_rmse_list_ffnn['rmse_c']=round(rmse_c, 3)
print(validation_dc_list_ffnn)   
plt.figure(figsize=(10, 6))
plt.plot(list(batch_dict.keys()), list(batch_dict.values()), marker='o', linestyle='-', color='b')
plt.xlabel('Batchsize')
plt.ylabel('R2 Score')
plt.title('R2 Score for Different Batch sizes')
plt.figure(figsize=(10, 6))
plt.plot(list(learning_rate_dict.keys()), list(learning_rate_dict.values()), marker='o', linestyle='-', color='b')
plt.xlabel('Learning Rate')
plt.ylabel('R2 Score')
plt.title('R2 Score for Different Learning Rates')

In [ ]:
validation_dc_list_lstm={}
calibration_dc_list_lstm={}
validation_cc_list_lstm={}
calibration_cc_list_lstm={}
validation_rmse_list_lstm={}
calibration_rmse_list_lstm={}
timestep_result={}
timesteps= [5]
for timestep in timesteps:
    x = gcms[sum(selected_columns.values(), [])]
    x['monthsin'],x['monthcos']=gcms['monthsin'],gcms['monthcos']
    y = gcms[selected_targets['approx']]
    real_observations=gcms['obs']
    real_observations_train,real_observations_test=train_test_split(real_observations,train_size=0.7,shuffle=False)
    xtrain, xtest, ytrain, ytest = train_test_split(x, y, train_size=0.7, shuffle=False)
    plt.figure(figsize=(15, 2))
    plt.plot(ytrain.index, ytrain, label='Train Data')
    plt.plot(ytest.index, ytest, label='Test Data')
    plt.legend()
    x_scaler = StandardScaler().fit(xtrain)
    y_scaler = StandardScaler().fit(np.array(ytrain).reshape(-1, 1))

    xtrain = x_scaler.transform(xtrain)
    xtest = x_scaler.transform(xtest)
    ytrain = y_scaler.transform(np.array(ytrain).reshape(-1, 1))
    ytest = y_scaler.transform(np.array(ytest).reshape(-1, 1))
    print("inputs-validation set normalized std dev:",np.std(ytest))
 
    
    xtrain_lstm, ytrain_lstm = prepare_lstm_input(xtrain, ytrain, timestep)
    xtest_lstm, ytest_lstm = prepare_lstm_input(xtest, ytest, timestep)
    print(f"xtrain_lstm shape: {xtrain_lstm.shape}, ytrain_lstm shape: {ytrain_lstm.shape}")
    print(f"xtest_lstm shape: {xtest_lstm.shape}, ytest_lstm shape: {ytest_lstm.shape}")
    lstm_model = keras.models.Sequential([
            keras.layers.LSTM(16, activation='selu', return_sequences=True, input_shape=(xtrain_lstm.shape[1], xtrain_lstm.shape[2]),kernel_regularizer=keras.regularizers.L1L2(0.01)),
            keras.layers.LSTM(32, activation='selu', return_sequences=True, input_shape=(xtrain_lstm.shape[1], xtrain_lstm.shape[2])),
            keras.layers.LSTM(16, activation='selu', return_sequences=False),
            keras.layers.Dropout(0.2),
            keras.layers.Dense(1)
        ])
        
        
    early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)
    lr_scheduler = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=30, verbose=1, min_lr=1e-5)
    lstm_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001) ,loss='mae', metrics=[NashSutcliffeEfficiency()],)
        
    history = lstm_model.fit(x=xtrain_lstm, y=ytrain_lstm, epochs=50, batch_size=16, validation_split=0.2, verbose=1,shuffle=False)
    plt.figure(figsize=(5, 5))
    plt.plot(history.history['loss'], label=f'Loss', alpha=0.6, linestyle=':')
    plt.plot(history.history['val_loss'], label='Validation Loss', alpha=0.5, linestyle='--')
    plt.style.use('dark_background')
    plt.legend()

    ypred = lstm_model.predict(xtest_lstm).flatten()
    ypred_train = lstm_model.predict(xtrain_lstm).flatten()
    ypred_transformed= y_scaler.inverse_transform(ypred.reshape(-1, 1))
    ytest_lstm_transformed= y_scaler.inverse_transform(ytest_lstm.reshape(-1, 1))
    print("Predictions-validation set normalized std dev:",np.std(ypred))
    print(f"Shapes - ytrain_lstm: {ytrain_lstm.shape}, ypred_train: {ypred_train.shape}")
    print(f"Shapes - ytest_lstm: {ytest_lstm.shape}, ypred: {ypred.shape}")
    
    dc_c = r2_score(y_true=ytrain_lstm, y_pred=ypred_train.reshape(-1, 1))
    coer_c = np.corrcoef(ypred_train, ytrain_lstm.reshape(-1))[0, 1]
    rmse_c = root_mean_squared_error(y_true=y_scaler.inverse_transform(ytrain_lstm.reshape(-1,1)), y_pred=y_scaler.inverse_transform(ypred_train.reshape(-1, 1)))
    
    dc_v = r2_score(y_true=ytest_lstm, y_pred=ypred.reshape(-1, 1))
    timestep_result[timestep]=dc_v
    coer_v = np.corrcoef(ypred, ytest_lstm.reshape(-1))[0, 1]
    rmse_v = root_mean_squared_error(y_true=y_scaler.inverse_transform(ytest_lstm.reshape(-1,1)), y_pred=y_scaler.inverse_transform(ypred.reshape(-1, 1)))
    
    plt.figure(figsize=(15, 5))
    plt.plot(index_test[:len(ypred)], ypred_transformed, label=f'Prediction', color='green')
    plt.plot(index_test[:len(ytest_lstm)],ytest_lstm_transformed, label=f'Target', color='yellow', alpha=0.5)
    plt.style.use('dark_background')
    plt.legend()
    
    print(f'DC Validation: {round(dc_v, 3)}, CC Validation: {round(coer_v, 3)}')
    print(f'DC Calibration: {round(dc_c, 3)}, CC Calibration: {round(coer_c, 3)}')
    
    validation_dc_list_lstm['dc_v'] = round(dc_v, 3)
    calibration_dc_list_lstm['dc_c'] = round(dc_c, 3)
    validation_cc_list_lstm['cc_v']=round(coer_v, 3)
    calibration_cc_list_lstm['cc_c']=round(coer_c, 3)
    validation_rmse_list_lstm['rmse_d']=round(rmse_v, 3)
    calibration_rmse_list_lstm['rmse_c']=round(rmse_c, 3)
    
    
print(validation_dc_list_lstm)
# Plot the dc_v score of each time step

plt.figure(figsize=(10, 6))
plt.plot(list(timestep_result.keys()), list(timestep_result.values()), marker='o', linestyle='-', color='b')
plt.xlabel('Timesteps')
plt.ylabel('R2 Score')
plt.title('R2 Score for Different Timesteps')
plt.grid(True)
plt.show()

In [ ]:

validation_dc_list_svr = {}
calibration_dc_list_svr = {}
validation_cc_list_svr = {}
calibration_cc_list_svr = {}
validation_rmse_list_svr={}
calibration_rmse_list_svr={}
x = gcms[sum(selected_columns.values(), [])]
x['monthsin'],x['monthcos']=gcms['monthsin'],gcms['monthcos']
y = gcms[selected_targets['approx']]

xtrain, xtest, ytrain, ytest = train_test_split(x, y, train_size=0.7, shuffle=False)
index_train, index_test = train_test_split(gcms.index, train_size=0.7, shuffle=False)
ytrain = np.array(ytrain).reshape(-1, 1)
ytest = np.array(ytest).reshape(-1, 1)
x_scaler = StandardScaler()
y_scaler = StandardScaler()

xtrain = x_scaler.fit_transform(xtrain)
xtest = x_scaler.transform(xtest)

ytrain = y_scaler.fit_transform(ytrain).reshape(-1)
ytest = y_scaler.transform(ytest).reshape(-1)

param_grid = {
    'C': [1,5, 10, 100, 200],
    'degree': [2, 3, 4],
    'gamma': ['scale', 0.1, 0.01, 0.001, 0.0001],
    'epsilon': [0.1, 0.01, 0.001, 0.0001, 0.0005],
    'kernel': ['rbf']
}
grid_search = GridSearchCV(svm.SVR(), param_grid, scoring='r2', cv=5)
grid_search.fit(xtrain, ytrain)
best_params = grid_search.best_params_
print(f'best parameters: {best_params}')

svr_model = svm.SVR(**best_params)
svr_model.fit(xtrain, ytrain)

ypred_svr = svr_model.predict(xtest)
ypred_train_svr = svr_model.predict(xtrain)
print("Predictions-validation set normalized std dev:",np.std(ypred_svr))
dc_c = r2_score(ytrain, ypred_train_svr)
dc_v = r2_score(ytest, ypred_svr)
coer_c = np.corrcoef(ytrain, ypred_train_svr)[0, 1]
coer_v = np.corrcoef(ytest, ypred_svr)[0, 1]
rmse_c = root_mean_squared_error(y_scaler.inverse_transform(ytrain.reshape(-1,1)),y_scaler.inverse_transform(ypred_train_svr.reshape(-1,1)))
rmse_v = root_mean_squared_error(y_scaler.inverse_transform(ytest.reshape(-1,1)), y_scaler.inverse_transform(ypred_svr.reshape(-1,1)))
plt.figure(figsize=(15, 5))
plt.plot(ypred_svr, label='Prediction', color='green')
plt.plot(ytest, label='Observations', color='blue', alpha=0.5)
plt.legend()

print(f'DC Validation: {dc_v:.3f}, CC Validation: {coer_v:.3f}')
print(f'DC Calibration: {dc_c:.3f}, CC Calibration: {coer_c:.3f}')

validation_dc_list_svr['dc_v'] = round(dc_v, 3)
calibration_dc_list_svr['dc_c'] = round(dc_c, 3)
validation_cc_list_svr['cc_v'] = round(coer_v, 3)
calibration_cc_list_svr['cc_c'] = round(coer_c, 3)
validation_rmse_list_svr['rmse_v']=round(rmse_v, 3)
calibration_rmse_list_svr['rmse_c']=round(rmse_c, 3)

In [ ]:

membfuncs = [
    {'function': 'sigmoid',
     'n_memb': 5,
     'params': {'c': {'value': [-1.0, -0.5, 0.0, 0.5, 1.0],  
                      'trainable': True},
                'gamma': {'value': [-1.0, -0.5, 0.0, 0.5, 1.0],  
                          'trainable': True}}}
    for _ in range(6)  
]

validation_dc_list_anfis = {}
calibration_dc_list_anfis = {}
validation_cc_list_anfis = {}
calibration_cc_list_anfis = {}
validation_rmse_list_anfis={}
calibration_rmse_list_anfis={}
x = gcms[selected_columns['r1i1p1f1']]
x['monthsin'],x['monthcos']=gcms['monthsin'],gcms['monthcos']
y = gcms[selected_targets['approx']]
x_scaler = StandardScaler()
y_scaler = StandardScaler()
x = x_scaler.fit_transform(x)
y = y_scaler.fit_transform(y.values.reshape(-1, 1))

xtrain, xtest, ytrain, ytest = train_test_split(x, y, train_size=0.7, shuffle=False)
xtrain_2, xval, ytrain_2, yval = train_test_split(xtrain, ytrain, test_size=0.3, shuffle=False)  
xtrain = np.nan_to_num(xtrain)
ytrain = np.nan_to_num(ytrain)
xtest = np.nan_to_num(xtest)
ytest = np.nan_to_num(ytest)
xval = np.nan_to_num(xval)
yval = np.nan_to_num(yval)
xtrain = torch.tensor(xtrain, dtype=torch.float32)
ytrain = torch.tensor(ytrain, dtype=torch.float32)
xtest = torch.tensor(xtest, dtype=torch.float32)
ytest = torch.tensor(ytest, dtype=torch.float32)
xval = torch.tensor(xval, dtype=torch.float32)
yval = torch.tensor(yval, dtype=torch.float32)


n_features = xtrain.shape[1]
fis = SANFIS(membfuncs=membfuncs, n_input=n_features, scale='Std')
loss_function = torch.nn.MSELoss(reduction='mean')
optimizer = torch.optim.Adam(fis.parameters(), lr=0.001)
fis_history = fis.fit([xtrain, ytrain], [xval, yval], optimizer, loss_function, epochs=80, batch_size=128)

plt.figure(figsize=(5, 5))
plt.plot(fis_history.train_curve, label=f'Train Curve')
plt.plot(fis_history.valid_curve, label='Validation Curve')
plt.legend()

ypred_anfis = fis.predict([xtest])
ypred_train_anfis = fis.predict([xtrain])
print("Predictions-validation set normalized std dev:",ypred_anfis.std())
dc_c = r2_score(y_pred=ypred_train_anfis.flatten().tolist(), y_true=ytrain.flatten().tolist())
coer_c = np.corrcoef(ypred_train_anfis.flatten().tolist(), ytrain.flatten().tolist())[0][1]
rmse_c = root_mean_squared_error(y_pred=y_scaler.inverse_transform(ypred_train_anfis.reshape(-1,1)).flatten().tolist(), y_true=y_scaler.inverse_transform(ytrain.reshape(-1,1)).flatten().tolist())
plt.figure(figsize=(15, 5))
plt.plot(ypred_anfis.flatten().tolist(), label=f'Prediction', color='green')
plt.plot( ytest.flatten().tolist(), label='Target', color='blue', alpha=0.5)
plt.legend()

dc_v = r2_score(y_pred=ypred_anfis.flatten().tolist(), y_true=ytest.flatten().tolist())
coer_v = np.corrcoef(ypred_anfis.flatten().tolist(), ytest.flatten().tolist())[0][1]
rmse_v = root_mean_squared_error(y_pred=y_scaler.inverse_transform(ypred_anfis.reshape(-1,1)).flatten().tolist(), y_true=y_scaler.inverse_transform(ytest.reshape(-1,1)).flatten().tolist())
print(f'DC Validation: {round(dc_v, 3)}, CC Validation: {round(coer_v, 3)}')
print(f'DC Calibration: {round(dc_c, 3)}, CC Calibration: {round(coer_c, 3)}')
validation_dc_list_anfis['dc_v'] = round(dc_v, 3)
calibration_dc_list_anfis["dc_c"] = round(dc_c, 3)
validation_cc_list_anfis['cc_v'] = round(coer_v, 3)
calibration_cc_list_anfis['cc_c'] = round(coer_c, 3)
validation_rmse_list_anfis['rmse_v']=round(rmse_v, 3)
calibration_rmse_list_anfis['rmse_c']=round(rmse_c, 3)

In [ ]:
predictions_dataframe= pd.DataFrame()
predictions_dataframe['ffnn']=np.concatenate([ypred_train_ffnn.reshape(-1),ypred_ffnn.reshape(-1)])
predictions_dataframe['svr']=np.concatenate([ypred_train_svr.reshape(-1),ypred_svr.reshape(-1)])
predictions_dataframe['anfis']=np.concatenate([ypred_train_anfis.reshape(-1),ypred_anfis.reshape(-1)])


In [ ]:

validation_dc_list_ensemble={}
calibration_dc_list_ensemble={}
validation_cc_list_ensemble={}
calibration_cc_list_ensemble={}
validation_rmse_list_ensemble={}
calibration_rmse_list_ensemble={}
batch_dict={}
batch_size = [64]
learning_rates = [0.001]
learning_rate_dict={}
for batch in batch_size:
    for lr in learning_rates:
        x = predictions_dataframe[['ffnn','svr','anfis']]
        y = gcms[selected_targets['approx']]
        xtrain, xtest, ytrain, ytest = train_test_split(x, y, train_size=0.7, shuffle=False)
        index_train, index_test = train_test_split(x.index, train_size=0.7, shuffle=False)
        ytrain = np.array(ytrain).reshape(-1, 1)
        ytest = np.array(ytest).reshape(-1, 1)
        y_scaler = StandardScaler()
        ytrain = y_scaler.fit_transform(ytrain).reshape(-1)
        ytest = y_scaler.transform(ytest).reshape(-1)
        print("inputs-validation set normalized std dev:",np.std(ytest))
        print(f'Input Shape: {xtrain.shape}\n target shape: {ytrain.shape}')
        
        ensemble_model = keras.models.Sequential([
                    keras.layers.Dense(16, activation=keras.activations.selu,kernel_initializer=keras.initializers.he_normal()),
                    keras.layers.Dropout(0.2),
              
                    keras.layers.Dense(1)
                ])
        ensemble_model.compile(loss=keras.losses.huber, optimizer=keras.optimizers.AdamW(learning_rate=lr,epsilon=1e-7), metrics=[NashSutcliffeEfficiency()])
        early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)
        lr_scheduler = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=30, verbose=1, min_lr=1e-5)
        
        ensemble_model.fit(x=xtrain, y=ytrain, epochs=50, batch_size=batch, validation_split=0.3,shuffle=False,callbacks=[lr_scheduler,early_stopping])
        plt.figure(figsize=(5, 5))
        plt.plot(ensemble_model.history.history['loss'], label=f'loss of  ')
        plt.plot(ensemble_model.history.history['val_loss'], label='val_Loss')
        plt.legend()
        
        ypred_ensemble = ensemble_model.predict(xtest)
        ypred_train_ensemble= ensemble_model.predict(xtrain)
        print("Predictions-validation set normalized std dev:",np.std(ypred_ensemble))
        dc_c = r2_score(y_true=ytrain, y_pred=ypred_train_ensemble)
        coer_c = np.corrcoef(ypred_train_ensemble.reshape(ypred_train_ensemble.shape[0]), ytrain.reshape(ytrain.shape[0]))[0, 1]
        rmse_c = root_mean_squared_error(y_true=y_scaler.inverse_transform(ytrain.reshape(-1,1)), y_pred=y_scaler.inverse_transform(ypred_train_ensemble.reshape(-1,1)))
        
        
        
        plt.figure(figsize=(15, 5))
        plt.plot( ypred_ensemble, label=f'Prediction', color='green')
        plt.plot( ytest, label=f'PRISM', color='blue',alpha=0.5)
        plt.legend()
        dc_v = r2_score(y_true=ytest, y_pred=ypred_ensemble)
        batch_dict[batch]=dc_v
        learning_rate_dict[lr]=dc_v
        coer_v = np.corrcoef(ypred_ensemble.reshape(ypred_ensemble.shape[0]), ytest.reshape(ytest.shape[0]))[0, 1]
        rmse_v = root_mean_squared_error(y_true=y_scaler.inverse_transform(ytest.reshape(-1,1)), y_pred=y_scaler.inverse_transform(ypred_ensemble.reshape(-1,1)))
        print(f'DC Validation:{round(dc_v, 3)}, CC Validation:{round(coer_v, 3)}')
        print(f'DC Calibration:{round(dc_c, 3)}, CC Calibration:{round(coer_c, 3)}')
        validation_dc_list_ensemble['dc_v'] = round(dc_v, 3)
        calibration_dc_list_ensemble['dc_c'] = round(dc_c, 3)
        validation_cc_list_ensemble['cc_v']=round(coer_v, 3)
        calibration_cc_list_ensemble['cc_c']=round(coer_c, 3)
        validation_rmse_list_ensemble['rmse_v']=round(rmse_v, 3)
        calibration_rmse_list_ensemble['rmse_c']=round(rmse_c, 3)
        print(validation_dc_list_ensemble)   
plt.figure(figsize=(10, 6))
plt.plot(list(batch_dict.keys()), list(batch_dict.values()), marker='o', linestyle='-', color='b')
plt.xlabel('Batchsize')
plt.ylabel('R2 Score')
plt.title('R2 Score for Different Batch sizes')
plt.figure(figsize=(10, 6))
plt.plot(list(learning_rate_dict.keys()), list(learning_rate_dict.values()), marker='o', linestyle='-', color='b')
plt.xlabel('Learning Rate')
plt.ylabel('R2 Score')
plt.title('R2 Score for Different Learning Rates')


In [ ]:
results = pd.DataFrame(columns=['ANFIS', 'SVR', 'FFNN', 'LSTM', 'Ensemble'], index=['DC', 'CC','RMSE'])
results.at['DC', 'FFNN'] = round(np.mean(list(validation_dc_list_ffnn.values())), 3)
results.at['DC', 'SVR'] = round(np.mean(list(validation_dc_list_svr.values())), 3)
results.at['DC', 'ANFIS'] = round(np.mean(list(validation_dc_list_anfis.values())), 3)
results.at['DC', 'LSTM'] = round(np.mean(list(validation_dc_list_lstm.values())), 3)
results.at['DC', 'Ensemble'] = round(np.mean(list(validation_dc_list_ensemble.values())), 3)

results.at['CC', 'FFNN'] = round(np.mean(list(validation_cc_list_ffnn.values())), 3)
results.at['CC', 'SVR'] = round(np.mean(list(validation_cc_list_svr.values())), 3)
results.at['CC', 'ANFIS'] = round(np.mean(list(validation_cc_list_anfis.values())), 3)
results.at['CC', 'LSTM'] = round(np.mean(list(validation_cc_list_lstm.values())), 3)
results.at['CC', 'Ensemble'] = round(np.mean(list(validation_cc_list_ensemble.values())), 3)

results.at['RMSE', 'FFNN'] = round(np.mean(list(validation_rmse_list_ffnn.values())), 3)
results.at['RMSE', 'SVR'] = round(np.mean(list(validation_rmse_list_svr.values())), 3)
results.at['RMSE', 'ANFIS'] = round(np.mean(list(validation_rmse_list_anfis.values())), 3)
results.at['RMSE', 'LSTM'] = round(np.mean(list(validation_rmse_list_lstm.values())), 3)
results.at['RMSE', 'Ensemble'] = round(np.mean(list(validation_rmse_list_ensemble.values())), 3)



In [ ]:
results.round(3)

In [ ]:
# Ensure index_test is in datetime format
index_test_datetime = pd.date_range(start='2004-10-01', periods=len(index_test), freq='M')

plt.figure(figsize=(15, 6)) 
plt.xlim(pd.Timestamp('2004-10-01'), pd.Timestamp('2014-12-01'))
plt.plot(index_test_datetime[:len(ypred_ensemble)], y_scaler.inverse_transform(ypred_ensemble.reshape(-1,1)), label='Ensemble', color='green', linestyle='-')
plt.plot(index_test_datetime[:len(ypred_svr)], y_scaler.inverse_transform(ypred_svr.reshape(-1,1)), label='SVR', color='blue', linestyle='--')
plt.plot(index_test_datetime[:len(ypred_ffnn)], y_scaler.inverse_transform(ypred_ffnn.reshape(-1,1)), label='FFNN', color='red', linestyle='-.')
plt.plot(index_test_datetime[:len(ypred_anfis)], y_scaler.inverse_transform(ypred_anfis.reshape(-1,1)), label='ANFIS', color='brown', linestyle=':')
plt.plot(index_test_datetime[-len(ypred):], y_scaler.inverse_transform(ypred.reshape(-1,1)), label='LSTM', color='purple', linestyle='dashdot')
plt.plot(index_test_datetime[:len(ytest)], y_scaler.inverse_transform(ytest.reshape(-1,1)), label='Observations', color='black', linestyle='solid',linewidth=5,alpha=0.3)
plt.style.use('default')
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"]=24
plt.ylabel('Precipitation(mm)')
#plt.legend(ncol=6)
plt.show()


In [ ]:
results.to_excel(f".\Sample Date\historical Inputs\{station}\{station}_best_deep_resutls.xlsx")



ffnn_model.save(f".\Sample Date\historical Inputs\{station}\ffnn_{station}.keras")
keras.saving.register_keras_serializable()

# Save the SANFIS model using pickle
with open(f".\Sample Date\historical Inputs\{station}\anfis\anfis_{station}.pickle", 'wb') as f:
	pickle.dump(fis, f)


# Save the SVR model using pickle
with open(f".\Sample Date\historical Inputs\{station}\svr\svr_model_{station}.pickle", 'wb') as f:
    pickle.dump(svr_model, f)
    
ensemble_model.save(f".\Sample Date\historical Inputs\{station}\ensemble\ensemble_{station}.keras")